# Adaptive Sampling


This notebook aims to reduce the number of samples required to train an accurate surrogate model. One approach is to use adaptive sampling, which explores the design space in under-sampled parts of the design space (exploration) while exploiting the knowledge gained from already finished simulation runs (exploitation).

This notebook is an example of the use of Lola-Voronoi based sampling. It is applicable in connection with any surrogate model type. Here it is applied with a Gaussian Process model. [[1]]

[1]: https://ieeexplore.ieee.org/abstract/document/5429687/

<img src="image/LOLA-Voronoi.jpg">

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from bdopt import EnergyPlusEvaluator, Problem, load_idf, sampling, viz
from bdopt.adaptive import AdaptiveSampler
from bdopt.surrogate import GPSurrogate

from parameter_sets import parameter_set

## Define sampler settings

In [ ]:
n_param = 7 # or 4 or 11
n_samples_init = 20 # initial set of samples collected using a Latin hypercube (low-discrepancy) design
no_iter = 5 # number of iterations the sampler runs
n = 4 # number of samples added per iteration of the adaptive sampler

n_test = n_samples_init // 5            # 20% of the initial samples are kept apart to measure the error of the surrogate model
n_initial = n_samples_init - n_test     # the remaining ones train the first surrogate model

## Set up parameters, objectives to set up a problem and an E+ evaluator instance.

For more details on that part check out the other notebooks of the course (`Besos_overview.ipynb`, `Surrogate_Modelling.ipynb`).

`parameter_set` (in `parameter_sets.py`) returns the set of 4, 7 or 11 parameters of the small office model `example_idf.idf`; each one is a bdopt `Parameter` with a unique name. In this model the window-to-wall ratio is limited to 0.1 to 0.45: its glass door stops the windows from growing further (see `window_to_wall.py`).

In [ ]:
parameters = parameter_set(n_param)
parameters

In [ ]:
problem = Problem(parameters, ['Electricity:Facility'])
idf = load_idf('example_idf.idf')
epw_file = 'ITA_Torino.160590_IWEC.epw'
evaluator = EnergyPlusEvaluator(problem, idf, epw_file,
                                out_dir='/tmp/bdopt_runs',   # work files go to the container's own disk (the shared folder is much slower)
                                err_dir='outputdir',         # runs that fail are kept here, with the EnergyPlus error file
                                processes=2)                 # EnergyPlus runs in parallel
print(evaluator.eplus)

## Initial training of Surrogate Model
Here we use a Gaussian Process surrogate model (`GPSurrogate`: one Gaussian process per objective, with a length scale for every parameter; the kernel is chosen by marginal likelihood, there is no grid search to run).

The sampler simulates the initial samples (a Latin hypercube, `n_initial` rows) and the held-out test samples (`n_test` rows), trains the surrogate on the first ones and measures its error on the test ones. With `max_iter=0` it stops right there: this is the *initial* surrogate. Every simulation is stored in the evaluator's cache, so the run below, which starts the same way, does not repeat them.

In [ ]:
sampler = AdaptiveSampler(problem, evaluator, method='flola_voronoi', surrogate=GPSurrogate(), seed=0)
initial = sampler.run(n_initial=n_initial, n_test=n_test, max_iter=0, tol=0)
initial.history.loc[0, ['n_train', 'scaled_rmse', 'r2:Electricity:Facility']]

`r2:Electricity:Facility` is the $R^2$ score on the held-out samples; `scaled_rmse` is the RMSE on them in units of the standard deviation of the objective.

# LOLA - Voronoi sampling
Here we run the LOLA-Voronoi sampling, in the *fuzzy* version of van der Herten et al. (`method='flola_voronoi'`; plain `method='lola_voronoi'` is also available, and many other criteria, see `bdopt.adaptive.METHODS`). New building designs to be simulated are picked around already simulated building designs with a high hybrid score $H$.

$H = V + E$

$H$ is used to incentivise exploration $V$ and exploitation $E$. $V$ is the Voronoi cell size to approximate the sample density and $E$ the local-linear estimate to approximate the gradient in the neighbourhood of a sample.

In [ ]:
result = sampler.run(n_initial=n_initial, n_test=n_test, max_iter=no_iter, batch=n, tol=0)
print(result)
print(evaluator.stats())

`result.data` holds every design that was simulated for training (first the initial ones, then the ones the sampler added, in order), `result.test` the held-out ones, `result.history` one row per iteration.

In [ ]:
result.history[['iteration', 'n_train', 'n_evaluations', 'scaled_rmse', 'r2:Electricity:Facility']]

## Visualization
We visualize the working scheme of the adaptive sampler by reducing the input dimensionality to 2 using multi-dimensional scaling.

In [ ]:
plt.plot(result.history['n_train'], result.history['r2:Electricity:Facility'], '-o')
plt.ylabel('R^2 score')
plt.xlabel('No. of samples')

In [ ]:
viz.convergence(result.history)
plt.show()

In [ ]:
# Manifold model
from sklearn.manifold import MDS
from sklearn.preprocessing import StandardScaler

P = problem.encode(result.data)                    # all the training designs, in the unit cube: the initial ones first
scaler_mds = StandardScaler()
p_norm = scaler_mds.fit_transform(P)

model = MDS(n_components=2, init='random', random_state=1)
out3 = model.fit_transform(p_norm)

plt.scatter(out3[:n_initial, 0], out3[:n_initial, 1], color='grey')
plt.scatter(out3[n_initial:, 0], out3[n_initial:, 1], color='r')
plt.axis('equal');

The density plot shows where the density of samples is high. Those areas may indicate more complex parts in the design space.

In [ ]:
from scipy.stats import gaussian_kde

new = out3[n_initial:]                              # the designs added by the adaptive sampler
span = np.abs(out3).max() * 1.1
gx, gy = np.meshgrid(np.linspace(-span, span, 100), np.linspace(-span, span, 100))
density = gaussian_kde(new.T)(np.vstack([gx.ravel(), gy.ravel()])).reshape(gx.shape)

plt.contourf(gx, gy, density, levels=10, cmap='Blues')
plt.scatter(new[:, 0], new[:, 1], color='r', s=12)
plt.ylim([-span, span])
plt.xlim([-span, span])

In [ ]:
from scipy.interpolate import griddata
x = out3[:, 0]
y = out3[:, 1]
z = result.surrogate.predict(result.data)['Electricity:Facility'].to_numpy()     # what the surrogate predicts at every design

# create x-y points to be used in heatmap
xi = np.linspace(x.min(), x.max(), 300)
yi = np.linspace(y.min(), y.max(), 300)

# zi is a matrix of x-y values
zi = griddata((x, y), z, (xi[None, :], yi[:, None]), method='cubic')

# the range of the colour bar: the range of the predictions
zmin, zmax = z.min(), z.max()
zi[(zi < zmin) | (zi > zmax)] = np.nan

# Create the contour plot
plt.figure(figsize=(10, 10))
CS = plt.contourf(xi, yi, zi, 10, cmap=plt.cm.rainbow,
                  vmax=zmax, vmin=zmin, alpha=.8)
plt.colorbar(label='Electricity:Facility predicted by the surrogate [J]')
plt.scatter(out3[:n_initial, 0], out3[:n_initial, 1], color='grey')
plt.scatter(out3[n_initial:, 0], out3[n_initial:, 1], color='r')

plt.grid()

## Is it better than a plain Latin hypercube? (optional)

Adaptive sampling does not always win: for smooth building responses a Latin hypercube with the same number of simulations is often as good, and the main use of the loop is to *know* when you have simulated enough. Compare on the held-out designs, which neither surrogate has seen. This cell simulates as many more designs as the adaptive run used for training (36 with the settings above), so it takes about as long as the whole run:

In [ ]:
lhs_designs = sampling.lhs(problem, len(result.data), seed=1)
lhs_data = evaluator.evaluate(lhs_designs, keep_input=True)
lhs_surrogate = GPSurrogate().fit(lhs_data, problem)

pd.DataFrame({'adaptive': result.surrogate.score(result.test)['rmse_pct'],
              'Latin hypercube': lhs_surrogate.score(result.test)['rmse_pct']})     # RMSE as % of the mean of the held-out values